# UHCL
this is an implementation of UHCL in pytorch. UHCL is a loss function that unifies represntation leaning, anomaly detection and hiearachical contrastive learning. 

## Intro
The set of all possible data-points within a space are called data-space. the following framework assumes that each given input-context (main dataset that we aim to develop an AI application for) is a subset of broader set which is data-space. the data-space is mostly noisy data-points, but there are data-points that are not noise nor in the input-context. this frame work creates a context hiearchy 
Noise --> outlier --> inlier (--> positive-inliers) and try to maintain a certain amount of similarity as we trace down the hiearchy tree.



In [67]:
#evaluate_summary
import os
import random
from typing import List, Tuple, Dict
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from collections import defaultdict 


# CONFIG
each experiment parameter is mentioned in this config to easily tinker with parameters, 

In [68]:
#
config = {
    "experiment_name": "standardcifar100vsboth", # Changed experiment name to avoid overwriting
    "inlier_dataset": "cifar100",
    "outlier_dataset": "both", #note : Use 'both' in order to expose the model to both cifar10 and svhn as outliers 

    
    "epochs":100,
    "batch_size": 100,
    "anchors_per_fine": 1,
    "lr": 1e-4,
    "min_lr" : 1e-6,
    "aux_weight": 0.000,
    "weight_decay": 0.000,   
    "alpha": 1/2,              # alpha[0] = alpha
    "alpha_decay": 1/2,         # alpha[j] = alpha * (alpha_decay ** j)
                                # the hierarchical design of UHCL ensures alpha[j] in each hiearchy and 
                                # it doesn't require calculation inside hiearchical loop
    "embed_dim": 128,
    "num_workers": 0,
    "pretrained": True,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "ckpt_dir": "./content/checkpoints",
#     "resume_ckpt_path": None, # If specified uses a .pth file to resume trainig
                              # if None it does the training from scracth (pretrained or random initilized)
    
    "resume_ckpt_path": "./content/checkpoints/standardcifar100vsboth_last.pth",
    
    
    # "mode": "supervised",   #TBA "supervised" or "selfsupervised" 
    # "use_noise": True,      #TBA
    # "balanced": False,      # if True: balanced sampling for MNIST family; TBA
}
os.makedirs(config["ckpt_dir"], exist_ok=True)


def set_seed(seed=59):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

class ToTensorSafe:
    def __call__(self, pic):
        if isinstance(pic, torch.Tensor):
            return pic.float() / 255. if pic.max() > 1 else pic.float()
        return transforms.functional.to_tensor(pic)

In [69]:
config.update({

    # ---------------------------------------------------------
    # Noise trajectory evaluation
    # ---------------------------------------------------------

    "noise_trajectory_steps": 50,

    # Number of CIFAR-100 test images used for trajectory.
    # 2000 is enough for a stable curve and keeps evaluation
    # reasonably fast.
    "noise_trajectory_samples": 2000,

    "noise_trajectory_seed": 35,

    # If 1.0 -> clean image.
    # If 0.0 -> pure Gaussian noise.
    "noise_trajectory_final_alpha_bar": 0.0,

    # ---------------------------------------------------------
    # Plotting
    # ---------------------------------------------------------

    "results_dir": "./results",

})

#  DATALOADERS


In [70]:

IMG_SIZE = {"mnist": 28, "fashionmnist": 28, "emnist": 28, "cifar10": 32, "cifar100": 32, "svhn": 32}

def get_transforms(name, mode="train"):
    size = IMG_SIZE.get(name, 32)
    tensorizer = ToTensorSafe()
    if name in ["mnist", "fashionmnist", "emnist"]:
        # grayscale datasets: no ImageNet norm
        if mode == "train":
            return transforms.Compose([tensorizer])
        else:
            return transforms.Compose([transforms.Resize(size), tensorizer])

    # color image datasets (CIFAR*)
    if mode == "train":
        return transforms.Compose([
        # MUST BE ENABLED for contrastive invariance
        transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        
        # Color augmentations
        transforms.ColorJitter(
            brightness=0.4,
            contrast=0.4,
            saturation=0.4,
            hue=0.1
        ),
        transforms.RandomGrayscale(p=0.2),
        
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        ),
    ])
    
    else:
        # Test-time: no augmentation except resizing + normalization
        return transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
        ])
class RandomNoiseDataset(Dataset):
      def __init__(self, num: int, shape: Tuple[int, int, int]):
            self.num = num; self.shape = shape
      def __len__(self): return self.num
      def __getitem__(self, idx): return torch.rand(self.shape), -1

In [71]:
import copy
import random
import numpy as np
import torch
from torch.utils.data import Subset

def get_random_coarse_split(dataset, split_ratio=0.5, seed=None):
    """
    Randomly selects a subset of coarse labels to be 'inliers'.
    Returns a set of inlier coarse labels.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    all_coarse_labels = list(dataset.coarse_label_indices.keys())
    random.shuffle(all_coarse_labels)
    
    n_inliers = int(len(all_coarse_labels) * split_ratio)
    inlier_labels = set(all_coarse_labels[:n_inliers])
    
    print(f"Split generated: {len(inlier_labels)} Inlier Coarse Classes")
    return inlier_labels

def split_dataset_by_labels(full_dataset, inlier_coarse_labels):
    """
    Splits a dataset into in/out sets based on a PRE-DEFINED set of coarse labels.
    """
    # 1. Identify Outlier Indices for the 'out_dataset'
    #    Iterate through all coarse labels in the dataset; if not in inlier list, it's an outlier.
    outlier_indices = []
    dataset_coarse_keys = full_dataset.coarse_label_indices.keys()
    
    for c_lbl in dataset_coarse_keys:
        if c_lbl not in inlier_coarse_labels:
            outlier_indices.extend(full_dataset.coarse_label_indices[c_lbl])
            
    out_dataset = Subset(full_dataset, outlier_indices)

    # 2. Prepare Inlier Dataset (Deep copy indices to avoid mutation issues)
    in_dataset = copy.copy(full_dataset)
    in_dataset.fine_label_indices = copy.deepcopy(full_dataset.fine_label_indices)
    in_dataset.coarse_label_indices = copy.deepcopy(full_dataset.coarse_label_indices)

    # Filter Coarse Indices: Keep only inlier keys
    in_dataset.coarse_label_indices = {
        c: idxs for c, idxs in in_dataset.coarse_label_indices.items() 
        if c in inlier_coarse_labels
    }

    # Filter Fine Indices: Keep only fine labels belonging to inlier coarse classes
    valid_fine_labels = []
    for fine_lbl, indices in in_dataset.fine_label_indices.items():
        if not indices: continue
        
        # Check coarse label of the first sample in this fine class
        # Access original dataset to get (x, fine, coarse) safely
        _, _, sample_coarse = full_dataset[indices[0]]
        
        if sample_coarse in inlier_coarse_labels:
            valid_fine_labels.append(fine_lbl)
            
    in_dataset.fine_label_indices = {
        f: in_dataset.fine_label_indices[f] for f in valid_fine_labels
    }

    return in_dataset, out_dataset

In [72]:
import torch
from torchvision import datasets
import numpy as np
from collections import defaultdict

class CIFAR100Hierarchy(datasets.CIFAR100):
    """
    Custom CIFAR-100 dataset that exposes both fine and coarse labels.
    CIFAR-100 has 20 coarse categories (superclasses), each containing 5 fine categories.
    Includes indexing for efficient sampling.
    """
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform,
                         target_transform=target_transform, download=download)

        # Define the hierarchy: mapping from fine labels to coarse labels
        # Each key is a coarse label, and the value is a list of fine labels in that category
        self.coarse_to_fine = {
            0: [4, 30, 55, 72, 95],   # aquatic mammals
            1: [1, 32, 67, 73, 91],   # fish
            2: [54, 62, 70, 82, 92],  # flowers
            3: [9, 10, 16, 28, 61],   # food containers
            4: [0, 51, 53, 57, 83],   # fruit and vegetables
            5: [22, 39, 40, 86, 87],  # household electrical devices
            6: [5, 20, 25, 84, 94],   # household furniture
            7: [6, 7, 14, 18, 24],    # insects
            8: [3, 42, 43, 88, 97],   # large carnivores
            9: [12, 17, 37, 68, 76],  # large man-made outdoor things
            10: [23, 33, 49, 60, 71], # large natural outdoor scenes
            11: [15, 19, 21, 31, 38], # large omnivores and herbivores
            12: [34, 63, 64, 66, 75], # medium-sized mammals
            13: [26, 45, 77, 79, 99], # non-insect invertebrates
            14: [2, 11, 35, 46, 98],  # people
            15: [27, 29, 44, 78, 93], # reptiles
            16: [36, 50, 65, 74, 80], # small mammals
            17: [47, 52, 56, 59, 96], # trees
            18: [8, 13, 48, 58, 90],  # vehicles 1
            19: [41, 69, 81, 85, 89], # vehicles 2
        }

        # Create a mapping from fine labels to coarse labels
        self.fine_to_coarse = {}
        for coarse_label, fine_labels in self.coarse_to_fine.items():
            for fine_label in fine_labels:
                self.fine_to_coarse[fine_label] = coarse_label

        # Create coarse_targets array for all samples
        self.coarse_targets = [self.fine_to_coarse[fine_label] for fine_label in self.targets]

        # Create indices for efficient sampling
        self.fine_label_indices = defaultdict(list)
        self.coarse_label_indices = defaultdict(list)

        for i, (fine_label, coarse_label) in enumerate(zip(self.targets, self.coarse_targets)):
            self.fine_label_indices[fine_label].append(i)
            self.coarse_label_indices[coarse_label].append(i)


    def __getitem__(self, index):
        """
        Args:
            index (int): Index
        Returns:
            tuple: (image, fine_target, coarse_target) where fine_target is the index of the target class.
        """
        img, fine_target = super().__getitem__(index)
        coarse_target = self.coarse_targets[index]
        return img, fine_target, coarse_target

class CIFAR10Indexed(datasets.CIFAR10):
    """
    Custom CIFAR-10 dataset with pre-computed indices for faster sampling.
    """
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform,
                         target_transform=target_transform, download=download)

        # Create indices for efficient sampling
        self.label_indices = defaultdict(list)
        for i, label in enumerate(self.targets):
            self.label_indices[label].append(i)

In [73]:
def sample_hierarchy_batch(in_dataset, out_dataset, noise_dataset,
                           batch_size, device, N = 4, is_hierarchical=False,
                           anchors_per_fine=config["anchors_per_fine"]):

    X_batch = []

    # ALWAYS create label containers
    yfine_batch = []
    ycoarse_batch = []   # will be None for CIFAR10
    y_batch = []         # CIFAR10 fine labels

    if is_hierarchical:
        fine_labels = list(in_dataset.fine_label_indices.keys())
        random.shuffle(fine_labels)

        for fine_label in fine_labels:
            fine_indices = in_dataset.fine_label_indices[fine_label]

            # anchors
            if len(fine_indices) < anchors_per_fine:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=True)
            else:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=False)

            for idx in anchors:
                try:
                    x0, fine0, coarse0 = in_dataset[idx]

                    # x1: same fine (pos)
                    pos_candidates = [i for i in in_dataset.fine_label_indices[fine0] if i != idx]
                    pos_idx = random.choice(pos_candidates) #if pos_candidates else idx
                    x1, _, _ = in_dataset[pos_idx]

                    # x2: same coarse, diff fine
                    coarse_candidates = [
                        i for i in in_dataset.coarse_label_indices[coarse0]
                        if in_dataset.targets[i] != fine0
                    ]
                    neg_fine_idx = random.choice(coarse_candidates) #if coarse_candidates else idx
                    x2, fine2, _ = in_dataset[neg_fine_idx]

                    # x3: diff coarse
                    all_coarse = list(in_dataset.coarse_label_indices.keys())
                    diff_coarse = [c for c in all_coarse if c != coarse0]
                    rand_coarse = random.choice(diff_coarse)
                    diff_coarse_candidates = in_dataset.coarse_label_indices[rand_coarse]
                    diff_coarse_idx = random.choice(diff_coarse_candidates)
                    x3, fine3, coarse3 = in_dataset[diff_coarse_idx]

                    # x4: outlier
                    # x4, _ = out_dataset[np.random.randint(len(out_dataset))]
                    x4, *rest = out_dataset[np.random.randint(len(out_dataset))]
                    # x5: noise
                    x5, _ = noise_dataset[np.random.randint(len(noise_dataset))]

                    # assemble tuple
                    X_tuple = torch.stack([x0, x1, x2, x3, x4, x5])

                    # labels ONLY for inlier-inlier comparisons (not outlier/not noise)
                    yfine_tuple = [fine0, fine0, fine2, fine3]    # 4 entries
                    ycoarse_tuple = [coarse0, coarse0, coarse0, coarse3]

                    X_batch.append(X_tuple)
                    yfine_batch.append(yfine_tuple)
                    ycoarse_batch.append(ycoarse_tuple)

                except:
                    print ('gotcha')
                    x0, _, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))

    else:
        # CIFAR10-like
        labels = list(in_dataset.label_indices.keys())
        random.shuffle(labels)

        for label in labels:
            label_indices = in_dataset.label_indices[label]

            if len(label_indices) < anchors_per_fine:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=True)
            else:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=False)

            for idx in anchors:
                try:
                    x0, y0 = in_dataset[idx]

                    pos_candidates = [i for i in in_dataset.label_indices[y0] if i != idx]
                    pos_idx = random.choice(pos_candidates) if pos_candidates else idx
                    x1, _ = in_dataset[pos_idx]

                    # diff label
                    diff_labels = [l for l in labels if l != y0]
                    neg_label = random.choice(diff_labels)
                    neg_idx = random.choice(in_dataset.label_indices[neg_label])
                    x2, y2 = in_dataset[neg_idx]

                    x3, _ = out_dataset[np.random.randint(len(out_dataset))]
                    x4, _ = noise_dataset[np.random.randint(len(noise_dataset))]

                    X_tuple = torch.stack([x0, x1, x2, x3, x4])
                    y_tuple = [y0, y0, y2]   # only for inliers

                    X_batch.append(X_tuple)
                    y_batch.append(y_tuple)

                except:
                    x0, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))

        # convert to tensors
        yfine_batch = y_batch                      # fine = class id
        ycoarse_batch = None                       # not used

    # clip size
    if len(X_batch) > batch_size:
        X_batch = X_batch[:batch_size]
        if yfine_batch is not None:
            yfine_batch = yfine_batch[:batch_size]
        if ycoarse_batch is not None:
            ycoarse_batch = ycoarse_batch[:batch_size]

    X_batch = torch.stack(X_batch).to(device)

    if ycoarse_batch is None:
        ycoarse_batch = torch.zeros((len(yfine_batch), 1), dtype=torch.long)

    return (
        X_batch,
        torch.tensor(yfine_batch, dtype=torch.long, device=device),
        torch.tensor(ycoarse_batch, dtype=torch.long, device=device),
    )


# MODEL & PROJECTION

In [74]:
class SmallResNet(nn.Module):

    def __init__(self, embed_dim=128, pretrained=False, in_channels=3, depth= 18, num_classes=None):
        super().__init__()
        if depth == 50:
            # Use ResNet50 weights if pretrained is True
            net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 2048  # ResNet50 outputs 2048 features before the final layer
        else: # Assuming depth 18 for other cases
            # Use ResNet18 weights if pretrained is True
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 512 # ResNet18 outputs 512 features before the final layer

        if in_channels != 3:
            # If the input channels are not 3, we need to modify the first convolutional layer.
            # We keep the original weights for the 3 input channels and initialize the new channels randomly.
            original_conv1 = net.conv1
            net.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
            if pretrained:
                 # Copy weights from the original 3 channels
                 net.conv1.weight.data[:, :3, :, :] = original_conv1.weight.data
                 # Initialize weights for additional channels randomly (e.g., Kaiming initialization)
                 nn.init.kaiming_normal_(net.conv1.weight.data[:, 3:, :, :], mode='fan_out', nonlinearity='relu')


        self.backbone = nn.Sequential(*list(net.children())[:-1])

        # Projector to map backbone features to desired embedding dimension
        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), # Add AdaptiveAvgPool2d to ensure 1x1 spatial dimensions
            nn.Flatten(), # Input: [batch_size, features, 1, 1], Output: [batch_size, features] (features = proj_in_features)
            nn.Linear(proj_in_features, 512), # Input: [batch_size, proj_in_features], Output: [batch_size, 512]
            nn.ReLU(inplace=True),            # Input: [batch_size, 512], Output: [batch_size, 512]
            nn.Linear(512, embed_dim),        # Input: [batch_size, 512], Output: [batch_size, embed_dim]
        )

        # Optional classification head for anomaly/noise detection
        self.clf_head = None
        if num_classes is not None:
            self.clf_head = nn.Linear(embed_dim, num_classes)

    def forward_features_and_logits(self, x):
        """
        Returns:
          emb:  [B, D]
          logits: [B, num_classes]   <- from clf_head (inlier/outlier/noise)
        """
        emb = self(x, return_embeddings=True)
    
        if self.clf_head is None:
            raise ValueError("clf_head is not defined. Make sure num_classes was set.")
    
        logits = self.clf_head(emb)
        return emb, logits

    
    #**** there might be a problem with last layer of my model, 
    #     double check if not including N outputs (classification probablities) in the output cause any error

    
    

    def forward(self, x, return_embeddings=True):
        z = self.backbone(x)
        z = self.projector(z)
        emb = F.normalize(z, p=2, dim=1)

        return emb, z
        

# UHCL

In [75]:
import torch
import torch.nn.functional as F

def uhcl(emb, N=5, alpha_base=0.5, alpha_decay=0.5, use_identity=False, device=None):
    """
    Unified Hierarchical Contrastive Loss (UHCL) - Explicit Margin Ordering
    
    Margin Order Guarantee:
      - Outer Boundary (Noise/Outlier, j = N-1): alpha_base (Largest = 0.5)
      - Inner Boundary (Fine Subclass, j = 1)  : alpha_inner (Smallest ~ 0.05)
    """
    if device is None:
        device = emb.device
        
    B, N, D = emb.shape
    assert N >= 3, f"UHCL requires at least 3 hierarchy levels, got {N}"
    
    # 1. Similarity Matrix S_{u, v} = <z_u, z_v>
    S = torch.bmm(emb, emb.transpose(1, 2))  # [B, N, N]
    total_loss = torch.tensor(0.0, device=device)
    
    # 2. Noise Orthogonality Constraint (<z_real, z_noise> <= 0)
    noise_sim = S[:, 0, N - 1]
    total_loss += F.relu(noise_sim).mean()
    
    # 3. Build Level Margins Explicitly
    # We create margins such that:
    #   alpha_by_level[N-1] = alpha_base (e.g., 0.5) [Outermost boundary]
    #   alpha_by_level[1]   = alpha_base * (alpha_decay ** (N-2)) [Innermost boundary]
    alpha_by_level = {}
    for j in range(1, N):
        # Distance from outer boundary: depth = (N - 1) - j
        depth = (N - 1) - j
        alpha_by_level[j] = alpha_base * (alpha_decay ** depth)
        
    # Verify Similarity Budget Constraint (Eq. 12: sum of inner margins <= 1.0)
    margin_sum = sum(alpha_by_level.values())
    if margin_sum > 1.0:
        scale = 1.0 / margin_sum
        for j in alpha_by_level:
            alpha_by_level[j] *= scale
            
    # 4. Hierarchical Hinge Optimization (Eq. 14, 15)
    hierarchy_loss = torch.tensor(0.0, device=device)
    
    for j in range(1, N):  # j = 1, ..., N-1
        # Fetch the exact level margin: j=1 gets smallest, j=N-1 gets largest
        alpha_j = alpha_by_level[j]
        level_loss = torch.tensor(0.0, device=device)
        
        for i in range(0, j):
            partial_loss = torch.tensor(0.0, device=device)
            
            for k in range(i, j):
                positive_sim = S[:, i, j - 1]  # S_{i, j-1}
                negative_sim = S[:, k, j]      # S_{k, j}
                
                # Hinge: max(S_{k, j} - S_{i, j-1} + alpha_j, 0)
                hinge = F.relu(negative_sim - positive_sim + alpha_j)
                partial_loss += hinge.mean()
            
            # Normalization factor lambda_{i, j} (Eq. 15)
            num_terms = j - i
            lambda_ij = 1.0 / num_terms
            level_loss += partial_loss * lambda_ij
        
        # Level weight lambda_j = 1 / j
        lambda_j = 1.0 / j
        hierarchy_loss += level_loss * lambda_j
    
    total_loss += hierarchy_loss
    return total_loss

# build experiment

In [76]:
def build_experiment(cfg):
    """
    Construct the datasets and model, but do NOT train anything.

    This function is shared by:
        - training
        - evaluation
    """

    device = cfg["device"]

    # =========================================================
    # 1. Transforms
    # =========================================================

    ttr = get_transforms(
        cfg["inlier_dataset"],
        "train"
    )

    tte = get_transforms(
        cfg["inlier_dataset"],
        "test"
    )

    # =========================================================
    # 2. Inlier dataset
    # =========================================================

    if cfg["inlier_dataset"] == "cifar100":

        in_train = CIFAR100Hierarchy(
            "./data",
            train=True,
            transform=ttr,
            download=True
        )

        in_test = CIFAR100Hierarchy(
            "./data",
            train=False,
            transform=tte,
            download=True
        )

        is_hierarchical = True
        N_levels_uhcl = 6
        depth = 50

    elif cfg["inlier_dataset"] == "cifar10":

        in_train = CIFAR10Indexed(
            "./data",
            train=True,
            transform=ttr,
            download=True
        )

        in_test = CIFAR10Indexed(
            "./data",
            train=False,
            transform=tte,
            download=True
        )

        is_hierarchical = False
        N_levels_uhcl = 5
        depth = 18

    else:

        raise ValueError(
            f"Unsupported inlier_dataset: "
            f"{cfg['inlier_dataset']}"
        )

    embed_dim = cfg["embed_dim"]

    # =========================================================
    # 3. Training OOD dataset
    # =========================================================

    print(
    f"Training OOD dataset: "
    f"{cfg['outlier_dataset']}"
    )

    if cfg["outlier_dataset"] == "svhn":

        out_train = datasets.SVHN(
            "./data",
            split="train",
            transform=ttr,
            download=True
        )

        out_test = datasets.SVHN(
            "./data",
            split="test",
            transform=tte,
            download=True
        )


    elif cfg["outlier_dataset"] == "cifar10":

        out_train = datasets.CIFAR10(
            "./data",
            train=True,
            transform=ttr,
            download=True
        )

        out_test = datasets.CIFAR10(
            "./data",
            train=False,
            transform=tte,
            download=True
        )


    elif cfg["outlier_dataset"] == "both":

        print(
            "Using CIFAR-10 + SVHN jointly as OOD."
        )

        # -----------------------------------------------------
        # CIFAR-10
        # -----------------------------------------------------

        cifar10_train = datasets.CIFAR10(
            "./data",
            train=True,
            transform=ttr,
            download=True
        )

        cifar10_test = datasets.CIFAR10(
            "./data",
            train=False,
            transform=tte,
            download=True
        )

        # -----------------------------------------------------
        # SVHN
        # -----------------------------------------------------

        svhn_train = datasets.SVHN(
            "./data",
            split="train",
            transform=ttr,
            download=True
        )

        svhn_test = datasets.SVHN(
            "./data",
            split="test",
            transform=tte,
            download=True
        )

        # -----------------------------------------------------
        # Balance the TRAINING OOD distributions
        #
        # CIFAR-10 has 50k training images while SVHN has more.
        # We use the same number from each.
        # -----------------------------------------------------

        n = min(
            len(cifar10_train),
            len(svhn_train)
        )

        rng = np.random.default_rng(42)

        cifar10_idx = rng.choice(
            len(cifar10_train),
            size=n,
            replace=False
        )

        svhn_idx = rng.choice(
            len(svhn_train),
            size=n,
            replace=False
        )

        cifar10_train = torch.utils.data.Subset(
            cifar10_train,
            cifar10_idx
        )

        svhn_train = torch.utils.data.Subset(
            svhn_train,
            svhn_idx
        )

        # -----------------------------------------------------
        # Combined OOD training dataset
        # -----------------------------------------------------

        out_train = torch.utils.data.ConcatDataset([
            cifar10_train,
            svhn_train,
        ])

        # -----------------------------------------------------
        # Keep the FULL test sets
        # -----------------------------------------------------

        out_test = torch.utils.data.ConcatDataset([
            cifar10_test,
            svhn_test,
        ])

        print(
            f"CIFAR-10 training OOD samples: {n}"
        )

        print(
            f"SVHN training OOD samples: {n}"
        )

        print(
            f"Combined OOD training samples: "
            f"{len(out_train)}"
        )

        print(
            f"Combined OOD test samples: "
            f"{len(out_test)}"
        )


    elif cfg["outlier_dataset"] == "cifar100":

        out_train = CIFAR100Hierarchy(
            "./data",
            train=True,
            transform=ttr,
            download=True
        )

        out_test = CIFAR100Hierarchy(
            "./data",
            train=False,
            transform=tte,
            download=True
        )


    else:

        raise ValueError(
            f"Unsupported outlier_dataset: "
            f"{cfg['outlier_dataset']}"
        )
        # -----------------------------------------------------
        # Combine the two OOD distributions
        # -----------------------------------------------------

        out_train = torch.utils.data.ConcatDataset([
            cifar10_train,
            svhn_train,
        ])

        out_test = torch.utils.data.ConcatDataset([
            cifar10_test,
            svhn_test,
        ])



    # =========================================================
    # 4. Noise datasets
    # =========================================================

    in_channels = (
        1
        if cfg["inlier_dataset"]
        in ["mnist", "fashionmnist", "emnist"]
        else 3
    )

    image_size = IMG_SIZE.get(
        cfg["inlier_dataset"],
        32
    )

    noise_train = RandomNoiseDataset(
        len(in_train),
        (
            in_channels,
            image_size,
            image_size
        )
    )

    noise_test = RandomNoiseDataset(
        len(in_test),
        (
            in_channels,
            image_size,
            image_size
        )
    )

    # =========================================================
    # 5. Model
    # =========================================================

    print(
        f"Initializing SmallResNet "
        f"with pretrained={cfg['pretrained']}"
    )

    model = SmallResNet(
        embed_dim=embed_dim,
        in_channels=in_channels,
        pretrained=cfg["pretrained"],
        depth=depth,
        num_classes=3,
    ).to(device)

    # =========================================================
    # 6. Dataset-specific classification heads
    # =========================================================

    if cfg["inlier_dataset"] == "cifar100":

        model.coarse_head = nn.Linear(
            embed_dim,
            20
        ).to(device)

        model.fine_head = nn.Linear(
            embed_dim,
            100
        ).to(device)

    elif cfg["inlier_dataset"] == "cifar10":

        model.inlier_head = nn.Linear(
            embed_dim,
            10
        ).to(device)

    return {
        "model": model,

        "in_train": in_train,
        "in_test": in_test,

        "out_train": out_train,
        "out_test": out_test,

        "noise_train": noise_train,
        "noise_test": noise_test,

        "is_hierarchical": is_hierarchical,
        "N_levels_uhcl": N_levels_uhcl,
        "embed_dim": embed_dim,
    }

# Train

In [77]:
def train(cfg):
    """
    Train the UHCL model.

    This function does NOT evaluate the model.
    """

    device = cfg["device"]

    exp = build_experiment(cfg)

    model = exp["model"]
    in_train = exp["in_train"]
    in_test = exp["in_test"]
    out_train = exp["out_train"]
    out_test = exp["out_test"]
    noise_train = exp["noise_train"]
    noise_test = exp["noise_test"]

    N_levels_uhcl = exp["N_levels_uhcl"]
    is_hierarchical = exp["is_hierarchical"]
    embed_dim = exp["embed_dim"]

    # =========================================================
    # Load checkpoint if requested
    # =========================================================

    resume_path = cfg.get(
        "resume_ckpt_path",
        None
    )

    if resume_path is not None:

        if not os.path.exists(resume_path):

            raise FileNotFoundError(
                f"Checkpoint not found:\n"
                f"{resume_path}"
            )

        print(
            f"Loading checkpoint: {resume_path}"
        )

        model.load_state_dict(
            torch.load(
                resume_path,
                map_location=device
            )
        )

        print(
            "Checkpoint loaded successfully."
        )

    # =========================================================
    # Training
    # =========================================================

    epochs = cfg["epochs"]

    if epochs <= 0:

        raise ValueError(
            "train() requires cfg['epochs'] > 0.\n"
            "For evaluation of an existing model, use "
            "evaluate_checkpoint() instead."
        )

    # ---------------------------------------------------------
    # Optimizer
    # ---------------------------------------------------------

    opt = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=epochs,
        eta_min=cfg["min_lr"]
    )

    # ---------------------------------------------------------
    # Main loader
    # ---------------------------------------------------------

    main_loader = DataLoader(
        in_train,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"]
    )

    # =========================================================
    # Main training loop
    # =========================================================

    for ep in range(epochs):

        model.train()

        epoch_loss = 0.0
        batch_count = 0

        pbar = tqdm(
            main_loader,
            desc=f"Epoch {ep + 1}/{epochs}"
        )

        for _ in pbar:

            # -------------------------------------------------
            # Sample hierarchy
            # -------------------------------------------------

            X, y_fine, y_coarse = sample_hierarchy_batch(
                in_train,
                out_train,
                noise_train,
                batch_size=cfg["batch_size"],
                device=device,
                N=N_levels_uhcl,
                is_hierarchical=is_hierarchical
            )

            if X.size(0) == 0:

                print(
                    "Warning: empty batch. Skipping."
                )

                continue

            # -------------------------------------------------
            # Forward
            # -------------------------------------------------

            B, N_levels, C, H, W = X.shape

            X = X.view(
                B * N_levels,
                C,
                H,
                W
            )

            Z_all, logits_all = model(X)

            Z_all = Z_all.view(
                B,
                N_levels,
                -1
            )

            logits_all = logits_all.view(
                B,
                N_levels,
                -1
            )

            # -------------------------------------------------
            # UHCL
            # -------------------------------------------------

            uhcl_loss = uhcl(
                Z_all,
                N=N_levels_uhcl,
                alpha_base=cfg["alpha"],
                alpha_decay=cfg["alpha_decay"],
                device=device
            )

            # -------------------------------------------------
            # Auxiliary classification
            # -------------------------------------------------

            if cfg["inlier_dataset"] == "cifar100":

                inlier_embs = (
                    logits_all[:, :4, :]
                    .reshape(-1, embed_dim)
                )

                fine_labels = (
                    y_fine[:, :4]
                    .reshape(-1)
                    .long()
                )

                coarse_labels = (
                    y_coarse[:, :4]
                    .reshape(-1)
                    .long()
                )

                fine_logits = model.fine_head(
                    inlier_embs
                )

                coarse_logits = model.coarse_head(
                    inlier_embs
                )

                fine_loss = F.cross_entropy(
                    fine_logits,
                    fine_labels
                )

                coarse_loss = F.cross_entropy(
                    coarse_logits,
                    coarse_labels
                )

                aux_loss = (
                    fine_loss + coarse_loss
                )

            else:

                inlier_embs = (
                    logits_all[:, :3, :]
                    .reshape(-1, embed_dim)
                )

                class_labels = (
                    y_fine[:, :3]
                    .reshape(-1)
                    .long()
                )

                inlier_logits = model.inlier_head(
                    inlier_embs
                )

                aux_loss = F.cross_entropy(
                    inlier_logits,
                    class_labels
                )

            # -------------------------------------------------
            # Total loss
            # -------------------------------------------------

            total_loss = (
                uhcl_loss
                + cfg["aux_weight"] * aux_loss
            )

            # -------------------------------------------------
            # Optimization
            # -------------------------------------------------

            opt.zero_grad()

            total_loss.backward()

            opt.step()

            # -------------------------------------------------
            # Statistics
            # -------------------------------------------------

            epoch_loss += uhcl_loss.item()
            batch_count += 1

            pbar.set_postfix({
                "UHCL": f"{uhcl_loss.item():.4f}",
                "Aux": f"{aux_loss.item():.4f}"
            })

        scheduler.step()

        if batch_count > 0:
            avg_loss = (
                epoch_loss / batch_count
            )
        else:
            avg_loss = float("nan")

        current_lr = (
            opt.param_groups[0]["lr"]
        )

        print(
            f"Epoch {ep + 1}/{epochs} "
            f"- UHCL: {avg_loss:.6f} "
            f"- LR: {current_lr:.6e}"
        )

        # -----------------------------------------------------
        # Periodic checkpoint
        # -----------------------------------------------------

        if (ep + 1) % 10 == 0:

            os.makedirs(
                cfg["ckpt_dir"],
                exist_ok=True
            )

            path = os.path.join(
                cfg["ckpt_dir"],
                f"{cfg['experiment_name']}"
                f"_ep{ep + 1}.pth"
            )

            torch.save(
                model.state_dict(),
                path
            )

            print(
                f"Checkpoint saved: {path}"
            )

    # =========================================================
    # Final checkpoint
    # =========================================================

    os.makedirs(
        cfg["ckpt_dir"],
        exist_ok=True
    )

    final_path = os.path.join(
        cfg["ckpt_dir"],
        f"{cfg['experiment_name']}_last.pth"
    )

    torch.save(
        model.state_dict(),
        final_path
    )

    print(
        "\nTraining complete."
    )

    print(
        f"Final checkpoint: {final_path}"
    )

    model.eval()

    return {
        "model": model,
        "in_train": in_train,
        "in_test": in_test,
        "out_train": out_train,
        "out_test": out_test,
        "noise_train": noise_train,
        "noise_test": noise_test,
    }



# EVAL ***

In [78]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
)


# ============================================================
# Utility
# ============================================================

def _classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
    }


def _normalize_np(x):
    """
    Row-wise L2 normalization.
    """
    x = np.asarray(x, dtype=np.float32)

    norms = np.linalg.norm(
        x,
        axis=1,
        keepdims=True
    )

    norms[norms == 0] = 1.0

    return x / norms


# ============================================================
# Extract both representations from the model
# ============================================================

@torch.no_grad()
def extract_eval_features(
    dataset,
    model,
    device,
    batch_size=256,
    hierarchical=False,
):
    """
    Extract frozen representations from the trained model.

    The current SmallResNet forward pass is assumed to return:

        z, head_features = model(images)

    where:
        z            = representation used by UHCL
        head_features = representation used by downstream
                        classification heads

    This function ONLY extracts features and labels.
    It does NOT perform classification or metric computation.
    """

    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    all_z = []
    all_head_features = []

    all_fine_labels = []
    all_coarse_labels = []

    for batch in loader:

        # =====================================================
        # Unpack batch
        # =====================================================

        if not isinstance(batch, (list, tuple)):
            raise ValueError(
                "Expected dataset batch to be a tuple/list."
            )

        images = batch[0]

        # Fine labels
        if len(batch) >= 2:
            fine_labels = batch[1]
        else:
            fine_labels = None

        # Coarse labels
        if hierarchical and len(batch) >= 3:
            coarse_labels = batch[2]
        else:
            coarse_labels = None

        # =====================================================
        # Move images to device
        # =====================================================

        images = images.to(device)

        # =====================================================
        # Model forward
        # =====================================================

        z, head_features = model(images)

        # =====================================================
        # Store representations
        # =====================================================

        all_z.append(
            z.detach().cpu()
        )

        all_head_features.append(
            head_features.detach().cpu()
        )

        # =====================================================
        # Store fine labels
        # =====================================================

        if fine_labels is not None:

            if not torch.is_tensor(fine_labels):
                fine_labels = torch.as_tensor(
                    fine_labels
                )

            all_fine_labels.append(
                fine_labels.detach().cpu().long()
            )

        # =====================================================
        # Store coarse labels
        # =====================================================

        if coarse_labels is not None:

            if not torch.is_tensor(coarse_labels):
                coarse_labels = torch.as_tensor(
                    coarse_labels
                )

            all_coarse_labels.append(
                coarse_labels.detach().cpu().long()
            )

    # =========================================================
    # Concatenate
    # =========================================================

    if len(all_z) == 0:

        raise RuntimeError(
            "Dataset produced no batches."
        )

    z = torch.cat(
        all_z,
        dim=0
    ).numpy()

    head_features = torch.cat(
        all_head_features,
        dim=0
    ).numpy()

    # =========================================================
    # Labels
    # =========================================================

    if all_fine_labels:

        fine_labels = torch.cat(
            all_fine_labels,
            dim=0
        ).numpy()

    else:

        fine_labels = None

    if all_coarse_labels:

        coarse_labels = torch.cat(
            all_coarse_labels,
            dim=0
        ).numpy()

    else:

        coarse_labels = None

    return (
        z,
        head_features,
        fine_labels,
        coarse_labels,
    )
# ============================================================
# Linear probe
# ============================================================

class LinearProbe(nn.Module):

    def __init__(
        self,
        feature_dim,
        num_classes,
    ):
        super().__init__()

        self.fc = nn.Linear(
            feature_dim,
            num_classes
        )

    def forward(self, x):
        return self.fc(x)


# ============================================================
# Angular / cosine probe
# ============================================================

class AngularProbe(nn.Module):
    """
    Cosine classifier.

    logits = scale * cosine(z, w)

    No angular margin is added.
    """

    def __init__(
        self,
        feature_dim,
        num_classes,
        scale=30.0,
    ):
        super().__init__()

        self.weight = nn.Parameter(
            torch.empty(
                num_classes,
                feature_dim
            )
        )

        nn.init.xavier_uniform_(
            self.weight
        )

        self.scale = scale

    def forward(self, x):

        x = F.normalize(
            x,
            p=2,
            dim=1
        )

        w = F.normalize(
            self.weight,
            p=2,
            dim=1
        )

        cosine = x @ w.T

        return self.scale * cosine


# ============================================================
# Fit a representation probe
# ============================================================

def fit_representation_probe(
    train_features,
    train_labels,
    num_classes,
    device,
    probe_type="linear",
    epochs=100,
    batch_size=256,
    lr=1e-3,
    weight_decay=0.0,
):
    """
    Fit a probe while keeping the backbone completely frozen.
    """

    x = torch.from_numpy(
        np.asarray(train_features)
    ).float()

    y = torch.from_numpy(
        np.asarray(train_labels)
    ).long()

    feature_dim = x.shape[1]

    if probe_type == "linear":

        probe = LinearProbe(
            feature_dim,
            num_classes
        ).to(device)

    elif probe_type == "angular":

        probe = AngularProbe(
            feature_dim,
            num_classes
        ).to(device)

    else:
        raise ValueError(
            "probe_type must be "
            "'linear' or 'angular'"
        )

    dataset = TensorDataset(
        x,
        y
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    optimizer = torch.optim.Adam(
        probe.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    criterion = nn.CrossEntropyLoss()

    probe.train()

    for epoch in range(epochs):

        total_loss = 0.0
        total_correct = 0
        total = 0

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            logits = probe(xb)

            loss = criterion(
                logits,
                yb
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += (
                loss.item() * yb.size(0)
            )

            pred = logits.argmax(
                dim=1
            )

            total_correct += (
                pred == yb
            ).sum().item()

            total += yb.size(0)

        if (
            epoch == 0
            or (epoch + 1) % 10 == 0
            or epoch == epochs - 1
        ):

            print(
                f"[{probe_type}] "
                f"Epoch {epoch + 1}/{epochs} "
                f"Loss={total_loss / total:.4f} "
                f"Acc={total_correct / total:.4f}"
            )

    return probe


@torch.no_grad()
def evaluate_representation_probe(
    probe,
    test_features,
    test_labels,
    device,
):
    probe.eval()

    x = torch.from_numpy(
        np.asarray(test_features)
    ).float().to(device)

    logits = probe(x)

    predictions = (
        logits
        .argmax(dim=1)
        .cpu()
        .numpy()
    )

    return (
        _classification_metrics(
            test_labels,
            predictions
        ),
        predictions
    )


# ============================================================
# KNN baseline
# ============================================================

def evaluate_knn(
    train_features,
    train_labels,
    test_features,
    test_labels,
    k=20,
):
    """
    Cosine KNN baseline.

    Only held-out test performance is treated as the
    manuscript metric.
    """

    train_features = _normalize_np(
        train_features
    )

    test_features = _normalize_np(
        test_features
    )

    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric="cosine",
    )

    knn.fit(
        train_features,
        train_labels
    )

    predictions = knn.predict(
        test_features
    )

    metrics = _classification_metrics(
        test_labels,
        predictions
    )

    return metrics, predictions


# ============================================================
# Post-hoc three-class probe
# ============================================================

class ThreeClassProbe(nn.Module):
    """
    0 = Inlier
    1 = Outlier
    2 = Noise
    """

    def __init__(
        self,
        feature_dim,
    ):
        super().__init__()

        self.fc = nn.Linear(
            feature_dim,
            3
        )

    def forward(self, x):
        return self.fc(x)


def fit_three_class_probe(
    in_features,
    out_features,
    noise_features,
    device,
    epochs=10,
    batch_size=256,
    lr=1e-3,
):
    """
    Post-hoc 3-class evaluation probe.

    The three classes are explicitly balanced using the
    smallest dataset size.

    IMPORTANT:
    These are frozen features. The backbone is NOT updated.
    """

    n = min(
        len(in_features),
        len(out_features),
        len(noise_features)
    )

    if n == 0:
        raise RuntimeError(
            "One of the 3-class evaluation datasets is empty."
        )

    rng = np.random.default_rng(42)

    in_idx = rng.choice(
        len(in_features),
        n,
        replace=False
    )

    out_idx = rng.choice(
        len(out_features),
        n,
        replace=False
    )

    noise_idx = rng.choice(
        len(noise_features),
        n,
        replace=False
    )

    X = np.concatenate([
        in_features[in_idx],
        out_features[out_idx],
        noise_features[noise_idx],
    ], axis=0)

    y = np.concatenate([
        np.zeros(n, dtype=np.int64),
        np.ones(n, dtype=np.int64),
        np.full(n, 2, dtype=np.int64),
    ])

    X = torch.from_numpy(
        X
    ).float()

    y = torch.from_numpy(
        y
    ).long()

    probe = ThreeClassProbe(
        X.shape[1]
    ).to(device)

    dataset = TensorDataset(
        X,
        y
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    optimizer = torch.optim.Adam(
        probe.parameters(),
        lr=lr
    )

    criterion = nn.CrossEntropyLoss()

    probe.train()

    for epoch in range(epochs):

        total_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            logits = probe(xb)

            loss = criterion(
                logits,
                yb
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += (
                loss.item() * yb.size(0)
            )

            pred = logits.argmax(
                dim=1
            )

            correct += (
                pred == yb
            ).sum().item()

            total += yb.size(0)

        print(
            f"[3-class probe] "
            f"Epoch {epoch + 1}/{epochs} "
            f"Loss={total_loss / total:.4f} "
            f"Acc={correct / total:.4f}"
        )

    return probe


@torch.no_grad()
def predict_three_class(
    probe,
    features,
    device,
):
    probe.eval()

    x = torch.from_numpy(
        np.asarray(features)
    ).float().to(device)

    logits = probe(x)

    probs = F.softmax(
        logits,
        dim=1
    )

    predictions = (
        probs.argmax(dim=1)
        .cpu()
        .numpy()
    )

    return (
        probs.cpu().numpy(),
        predictions
    )


# ============================================================
# OOD metrics
# ============================================================

def evaluate_ood_scores(
    id_scores,
    ood_scores,
):
    """
    Positive class = OOD.
    Higher score means more OOD.
    """

    y_true = np.concatenate([
        np.zeros(
            len(id_scores),
            dtype=np.int64
        ),
        np.ones(
            len(ood_scores),
            dtype=np.int64
        ),
    ])

    y_score = np.concatenate([
        id_scores,
        ood_scores,
    ])

    auroc = roc_auc_score(
        y_true,
        y_score
    )

    aupr = average_precision_score(
        y_true,
        y_score
    )

    # --------------------------------------------------------
    # FPR @ 95% OOD recall
    # --------------------------------------------------------

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_score
    )

    valid = np.where(
        tpr >= 0.95
    )[0]

    if len(valid) > 0:

        idx = valid[0]

        fpr95 = fpr[idx]
        threshold95 = thresholds[idx]

    else:

        fpr95 = np.nan
        threshold95 = np.nan

    return {
        "AUROC": float(auroc),
        "AUPR_OOD": float(aupr),
        "FPR_at_95_OOD_recall": float(fpr95),
        "threshold_at_95_OOD_recall": float(
            threshold95
        ),
    }


# ============================================================
# Full manuscript evaluation
# ============================================================
def load_checkpoint_for_evaluation(
    cfg,
    checkpoint_path,
):
    """
    Build the experiment architecture/datasets and load an
    already-trained checkpoint.

    NO training is performed.
    """

    device = cfg["device"]

    exp = build_experiment(cfg)

    model = exp["model"]

    if not os.path.exists(checkpoint_path):

        raise FileNotFoundError(
            f"Checkpoint not found:\n"
            f"{checkpoint_path}"
        )

    print(
        f"\nLoading checkpoint for evaluation:\n"
        f"{checkpoint_path}"
    )

    state_dict = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        state_dict
    )

    model.eval()

    print(
        "Checkpoint loaded."
    )

    print(
        "Backbone set to evaluation mode."
    )

    exp["model"] = model

    return exp

# I commented the following because there seemed to be a duplicate of evaluate_manuscript
# def evaluate_manuscript(
#     cfg,
#     exp,
# ):
#     """
#     Manuscript evaluation of an already-trained model.
#     """

#     model = exp["model"]

#     in_train = exp["in_train"]
#     in_test = exp["in_test"]

#     out_train = exp["out_train"]

#     noise_train = exp["noise_train"]
#     noise_test = exp["noise_test"]

#     device = cfg["device"]


#     batch_size = config.get(
#         "eval_batch_size",
#         256
#     )

#     probe_epochs = config.get(
#         "probe_epochs",
#         100
#     )

#     # ========================================================
#     # Validate experiment
#     # ========================================================

#     if config["inlier_dataset"] != "cifar100":

#         raise ValueError(
#             "This manuscript evaluation protocol assumes "
#             "CIFAR-100 is the inlier dataset, because CIFAR-10 "
#             "and SVHN are being evaluated as near/far OOD."
#         )

#     print(
#         "\n=================================================="
#     )

#     print(
#         "MANUSCRIPT EVALUATION"
#     )

#     print(
#         "=================================================="
#     )

#     print(
#         "Inlier: CIFAR-100"
#     )

#     print(
#         "Near OOD: CIFAR-10"
#     )

#     print(
#         "Far OOD: SVHN"
#     )

#     print(
#         f"OOD distribution used to fit 3-class probe: "
#         f"{config['outlier_dataset']}"
#     )

#     # ========================================================
#     # 1. Build CIFAR-10 and SVHN datasets independently
#     # ========================================================

#     test_transform = get_transforms(
#         "cifar100",
#         "test"
#     )

#     cifar10_test = datasets.CIFAR10(
#         "./data",
#         train=False,
#         transform=test_transform,
#         download=True
#     )

#     svhn_test = datasets.SVHN(
#         "./data",
#         split="test",
#         transform=test_transform,
#         download=True
#     )

#     # ========================================================
#     # 2. Build the training OOD dataset for post-hoc
#     #    3-class probe
#     # ========================================================

def evaluate_manuscript(
    cfg,
    exp,
):
    """
    Main manuscript evaluation.

    Returns ONLY the paper-facing evaluation results:

        representation
        inlier_classification
        ood
        three_class
        noise

    Progressive noise-trajectory measurements are deliberately NOT included
    here. They are run and saved separately so the main manuscript result
    object stays compact and stable.
    """

    model = exp["model"]
    in_train = exp["in_train"]
    in_test = exp["in_test"]
    out_train = exp["out_train"]
    noise_train = exp["noise_train"]
    noise_test = exp["noise_test"]
    device = cfg["device"]

    batch_size = cfg.get("eval_batch_size", 256)
    probe_epochs = cfg.get("probe_epochs", 100) #changed from 10
    three_class_probe_epochs = cfg.get("three_class_probe_epochs", 100) #changed from 30

    if cfg["inlier_dataset"] != "cifar100":
        raise ValueError(
            "This manuscript evaluation currently assumes CIFAR-100 as the inlier dataset."
        )

    print("\n==================================================")
    print("MANUSCRIPT EVALUATION")
    print("==================================================")
    print("Inlier: CIFAR-100")
    print("Near OOD: CIFAR-10")
    print("Far OOD: SVHN")
    print(f"OOD distribution used to train 3-class probe: {cfg['outlier_dataset']}")

    # ---------------------------------------------------------
    # Independent held-out OOD test sets
    # ---------------------------------------------------------
    test_transform = get_transforms("cifar100", "test")

    cifar10_test = datasets.CIFAR10(
        "./data", train=False, transform=test_transform, download=True
    )
    svhn_test = datasets.SVHN(
        "./data", split="test", transform=test_transform, download=True
    )

    # ---------------------------------------------------------
    # Frozen feature extraction
    # ---------------------------------------------------------
    print("\n--- Extracting CIFAR-100 train features ---")
    train_z, train_head_features, train_fine, train_coarse = extract_eval_features(
        in_train, model, device, batch_size=batch_size, hierarchical=True
    )

    print("\n--- Extracting CIFAR-100 test features ---")
    test_z, test_head_features, test_fine, test_coarse = extract_eval_features(
        in_test, model, device, batch_size=batch_size, hierarchical=True
    )

    print("\n--- Extracting OOD TRAIN features ---")
    _, out_train_head_features, _, _ = extract_eval_features(
        out_train, model, device, batch_size=batch_size, hierarchical=False
    )

    print("\n--- Extracting noise TRAIN features ---")
    _, noise_train_head_features, _, _ = extract_eval_features(
        noise_train, model, device, batch_size=batch_size, hierarchical=False
    )

    print("\n--- Extracting noise TEST features ---")
    _, noise_test_head_features, _, _ = extract_eval_features(
        noise_test, model, device, batch_size=batch_size, hierarchical=False
    )

    print("\n--- Extracting CIFAR-10 TEST features ---")
    cifar10_z, cifar10_head_features, _, _ = extract_eval_features(
        cifar10_test, model, device, batch_size=batch_size, hierarchical=False
    )

    print("\n--- Extracting SVHN TEST features ---")
    svhn_z, svhn_head_features, _, _ = extract_eval_features(
        svhn_test, model, device, batch_size=batch_size, hierarchical=False
    )

    results = {
        "representation": {},
        "inlier_classification": {},
        "ood": {},
        "three_class": {},
        "noise": {},
    }

    # ---------------------------------------------------------
    # Representation probes
    # ---------------------------------------------------------
    print("\n--- Representation evaluation ---")

    metrics, _ = evaluate_knn(
        train_z, train_fine, test_z, test_fine, k=cfg.get("knn_k", 20)
    )
    results["representation"]["knn_fine"] = metrics
    print("KNN Fine:", metrics)

    metrics, _ = evaluate_knn(
        train_z, train_coarse, test_z, test_coarse, k=cfg.get("knn_k", 20)
    )
    results["representation"]["knn_coarse"] = metrics
    print("KNN Coarse:", metrics)

    linear_fine_probe = fit_representation_probe(
        train_z, train_fine, 100, device, probe_type="linear",
        epochs=probe_epochs, batch_size=batch_size,
        lr=cfg.get("probe_lr", 1e-3),
        weight_decay=cfg.get("probe_weight_decay", 0.0),
    )
    metrics, _ = evaluate_representation_probe(
        linear_fine_probe, test_z, test_fine, device
    )
    results["representation"]["linear_fine"] = metrics
    print("Linear Fine:", metrics)

    linear_coarse_probe = fit_representation_probe(
        train_z, train_coarse, 20, device, probe_type="linear",
        epochs=probe_epochs, batch_size=batch_size,
        lr=cfg.get("probe_lr", 1e-3),
        weight_decay=cfg.get("probe_weight_decay", 0.0),
    )
    metrics, _ = evaluate_representation_probe(
        linear_coarse_probe, test_z, test_coarse, device
    )
    results["representation"]["linear_coarse"] = metrics
    print("Linear Coarse:", metrics)

    angular_fine_probe = fit_representation_probe(
        train_z, train_fine, 100, device, probe_type="angular",
        epochs=probe_epochs, batch_size=batch_size,
        lr=cfg.get("probe_lr", 1e-3),
        weight_decay=cfg.get("probe_weight_decay", 0.0),
    )
    metrics, _ = evaluate_representation_probe(
        angular_fine_probe, test_z, test_fine, device
    )
    results["representation"]["angular_fine"] = metrics
    print("Angular Fine:", metrics)

    angular_coarse_probe = fit_representation_probe(
        train_z, train_coarse, 20, device, probe_type="angular",
        epochs=probe_epochs, batch_size=batch_size,
        lr=cfg.get("probe_lr", 1e-3),
        weight_decay=cfg.get("probe_weight_decay", 0.0),
    )
    metrics, _ = evaluate_representation_probe(
        angular_coarse_probe, test_z, test_coarse, device
    )
    results["representation"]["angular_coarse"] = metrics
    print("Angular Coarse:", metrics)

    # ---------------------------------------------------------
    # Direct trained CIFAR-100 classification heads
    # ---------------------------------------------------------
    print("\n--- Direct CIFAR-100 classification heads ---")

    test_head_tensor = torch.from_numpy(
        np.asarray(test_head_features)
    ).float().to(device)

    if hasattr(model, "fine_head"):
        fine_logits = model.fine_head(test_head_tensor)
        fine_pred = fine_logits.argmax(dim=1).cpu().numpy()
        results["inlier_classification"]["fine"] = _classification_metrics(
            test_fine, fine_pred
        )
        print("Fine:", results["inlier_classification"]["fine"])

    if hasattr(model, "coarse_head"):
        coarse_logits = model.coarse_head(test_head_tensor)
        coarse_pred = coarse_logits.argmax(dim=1).cpu().numpy()
        results["inlier_classification"]["coarse"] = _classification_metrics(
            test_coarse, coarse_pred
        )
        print("Coarse:", results["inlier_classification"]["coarse"])

    # ---------------------------------------------------------
    # Post-hoc 3-class probe
    # ---------------------------------------------------------
    print("\n==================================================")
    print("Training post-hoc Inlier / Outlier / Noise probe")
    print("==================================================")

    three_class_probe = fit_three_class_probe(
        in_features=train_head_features,
        out_features=out_train_head_features,
        noise_features=noise_train_head_features,
        device=device,
        epochs=three_class_probe_epochs,
        batch_size=batch_size,
        lr=cfg.get("three_class_probe_lr", 1e-3),
    )

    print("3-class probe training complete.")

    id_probs, id_pred = predict_three_class(
        three_class_probe, test_head_features, device
    )
    cifar10_probs, cifar10_pred = predict_three_class(
        three_class_probe, cifar10_head_features, device
    )
    svhn_probs, svhn_pred = predict_three_class(
        three_class_probe, svhn_head_features, device
    )
    noise_test_probs, noise_test_pred = predict_three_class(
        three_class_probe, noise_test_head_features, device
    )

    # ---------------------------------------------------------
    # OOD + 3-class + static noise metrics
    # ---------------------------------------------------------
    ood_datasets = {
        "CIFAR10": {
            "probs": cifar10_probs,
            "pred": cifar10_pred,
            "relation": "near_OOD",
        },
        "SVHN": {
            "probs": svhn_probs,
            "pred": svhn_pred,
            "relation": "far_OOD",
        },
    }

    training_ood = cfg["outlier_dataset"].lower()

    for name, info in ood_datasets.items():
        probs = info["probs"]
        ood_pred = info["pred"]

        seen_during_probe_training = (
            True if training_ood == "both" else name.lower() == training_ood
        )

        id_ood_score = 1.0 - id_probs[:, 0]
        ood_ood_score = 1.0 - probs[:, 0]
        generic_ood_metrics = evaluate_ood_scores(
            id_ood_score, ood_ood_score
        )

        specific_outlier_metrics = evaluate_ood_scores(
            id_probs[:, 1], probs[:, 1]
        )

        y_true = np.concatenate([
            np.zeros(len(id_pred), dtype=np.int64),
            np.ones(len(ood_pred), dtype=np.int64),
            np.full(len(noise_test_pred), 2, dtype=np.int64),
        ])
        y_pred = np.concatenate([
            id_pred,
            ood_pred,
            noise_test_pred,
        ])

        three_class_metrics = _classification_metrics(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

        static_noise_metrics = evaluate_ood_scores(
            id_probs[:, 2], noise_test_probs[:, 2]
        )

        results["ood"][name] = {
            "relation": info["relation"],
            "seen_during_probe_training": seen_during_probe_training,
            "generic_ood_score": generic_ood_metrics,
            "specific_outlier_probability_score": specific_outlier_metrics,
        }

        results["three_class"][name] = {
            **three_class_metrics,
            "confusion_matrix": cm.tolist(),
        }

        results["noise"][name] = static_noise_metrics

        print(f"\n{name} ({info['relation']})")
        print("Seen during probe training:", seen_during_probe_training)
        print(f"Generic OOD AUROC: {generic_ood_metrics['AUROC']:.4f}")
        print(f"Generic OOD AUPR: {generic_ood_metrics['AUPR_OOD']:.4f}")
        print(
            f"Generic OOD FPR@95: "
            f"{generic_ood_metrics['FPR_at_95_OOD_recall']:.4f}"
        )
        print(
            f"Specific P(outlier) AUROC: "
            f"{specific_outlier_metrics['AUROC']:.4f}"
        )
        print(f"3-class Accuracy: {three_class_metrics['accuracy']:.4f}")
        print(f"3-class Macro F1: {three_class_metrics['f1_macro']:.4f}")
        print(f"Noise AUROC: {static_noise_metrics['AUROC']:.4f}")

    return results

#*******************************************************************
def make_noise_trajectory(
    x0,
    steps=10,
    final_alpha_bar=0.0,
    generator=None,
):
    """
    Create a controlled Gaussian-noise trajectory.

    x_t = sqrt(alpha_bar_t) * x0
          + sqrt(1 - alpha_bar_t) * epsilon

    The SAME epsilon is used for every step so that each
    sample follows a smooth trajectory from clean -> noise.

    Parameters
    ----------
    x0:
        Tensor [B, C, H, W].

    steps:
        Number of corruption steps.

    final_alpha_bar:
        Signal power at the final step.
        1.0 = no corruption
        0.0 = pure Gaussian noise

    generator:
        Optional torch.Generator for reproducibility.

    Returns
    -------
    trajectory:
        List of tensors:
        [x_0, x_1, ..., x_T]

    alpha_bars:
        Corresponding signal powers.
    """

    if steps < 1:
        raise ValueError(
            "steps must be >= 1"
        )

    if not (
        0.0 <= final_alpha_bar <= 1.0
    ):
        raise ValueError(
            "final_alpha_bar must be in [0, 1]"
        )

    # ---------------------------------------------------------
    # Smooth cosine schedule
    #
    # alpha_bar(0) = 1
    # alpha_bar(T) = final_alpha_bar
    # ---------------------------------------------------------

    t = torch.linspace(
        0.0,
        1.0,
        steps + 1,
        device=x0.device
    )

    cosine_signal = (
        torch.cos(
            0.5 * np.pi * t
        ) ** 2
    )

    alpha_bars = (
        final_alpha_bar
        + (1.0 - final_alpha_bar)
        * cosine_signal
    )

    # ---------------------------------------------------------
    # One fixed noise realization per image
    # ---------------------------------------------------------

    if generator is None:

        epsilon = torch.randn_like(
            x0
        )

    else:

        epsilon = torch.randn(
            x0.shape,
            device=x0.device,
            dtype=x0.dtype,
            generator=generator,
        )

    trajectory = []

    for alpha_bar in alpha_bars:

        signal = torch.sqrt(
            alpha_bar
        )

        noise = torch.sqrt(
            1.0 - alpha_bar
        )

        xt = (
            signal * x0
            + noise * epsilon
        )

        trajectory.append(
            xt
        )

    return (
        trajectory,
        alpha_bars.detach().cpu().numpy()
    )
def collect_eval_images(
    dataset,
    device,
    num_samples=2000,
    batch_size=256,
):
    n = min(
        len(dataset),
        num_samples
    )

    subset = torch.utils.data.Subset(
        dataset,
        np.arange(n)
    )

    loader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    images = []
    fine_labels = []
    coarse_labels = []

    for batch in loader:
        images.append(batch[0])
        fine_labels.append(batch[1])

        if len(batch) >= 3:
            coarse_labels.append(batch[2])

    images = torch.cat(
        images,
        dim=0
    )

    fine_labels = torch.cat(
        fine_labels,
        dim=0
    ).numpy()

    if coarse_labels:
        coarse_labels = torch.cat(
            coarse_labels,
            dim=0
        ).numpy()
    else:
        coarse_labels = None

    return (
        images.to(device),
        fine_labels,
        coarse_labels,
    )


@torch.no_grad()
def evaluate_noise_trajectory(
    model,
    inlier_test_set,
    device,
    three_class_probe=None,
    num_samples=2000,
    steps=10,
    batch_size=256,
    final_alpha_bar=0.0,
    seed=42,
):
    """
    Progressive clean -> Gaussian-noise evaluation.

    The same Gaussian realization is used at every step, producing
    a smooth trajectory. The post-hoc 3-class probe is used for
    P(Inlier), P(Outlier), and P(Noise), while the trained CIFAR-100
    heads provide fine/coarse semantic accuracy.
    """

    model.eval()

    if three_class_probe is not None:
        three_class_probe.eval()

    x0, fine_labels, coarse_labels = collect_eval_images(
        inlier_test_set,
        device,
        num_samples=num_samples,
        batch_size=batch_size,
    )

    n = x0.size(0)

    generator = torch.Generator(device=device)
    generator.manual_seed(seed)

    trajectory, alpha_bars = make_noise_trajectory(
        x0,
        steps=steps,
        final_alpha_bar=final_alpha_bar,
        generator=generator,
    )

    clean_z = []

    for start in range(0, n, batch_size):
        xb = x0[
            start:start + batch_size
        ]

        z, _ = model(xb)

        clean_z.append(
            F.normalize(
                z,
                p=2,
                dim=1
            )
        )

    clean_z = torch.cat(
        clean_z,
        dim=0
    )

    results = {
        "step": [],
        "alpha_bar": [],
        "signal_fraction": [],
        "noise_fraction": [],

        "mean_p_inlier": [],
        "mean_p_outlier": [],
        "mean_p_noise": [],

        "predicted_inlier_fraction": [],
        "predicted_outlier_fraction": [],
        "predicted_noise_fraction": [],

        "mean_clean_cosine_similarity": [],

        "fine_accuracy": [],
        "coarse_accuracy": [],
    }

    for step_idx, xt in enumerate(trajectory):

        all_probs = []
        all_cosine = []

        all_fine_correct = 0
        all_coarse_correct = 0
        all_fine_count = 0
        all_coarse_count = 0

        for start in range(0, n, batch_size):

            xb = xt[
                start:start + batch_size
            ]

            clean_zb = clean_z[
                start:start + batch_size
            ]

            z, head_features = model(xb)

            # Embedding geometry.
            z_norm = F.normalize(
                z,
                p=2,
                dim=1
            )

            cosine_sim = (
                z_norm * clean_zb
            ).sum(dim=1)

            all_cosine.append(
                cosine_sim.cpu()
            )

            # Post-hoc 3-class probabilities.
            if three_class_probe is not None:
                clf_logits = three_class_probe(
                    head_features
                )

                probs = F.softmax(
                    clf_logits,
                    dim=1
                )

                all_probs.append(
                    probs.cpu()
                )

            # Fine classification.
            y_fine = torch.as_tensor(
                fine_labels[
                    start:start + batch_size
                ],
                device=device
            )

            if hasattr(model, "fine_head"):
                fine_logits = model.fine_head(
                    head_features
                )

                fine_pred = (
                    fine_logits
                    .argmax(dim=1)
                )

                all_fine_correct += (
                    fine_pred == y_fine
                ).sum().item()

                all_fine_count += y_fine.size(0)

            # Coarse classification.
            if (
                coarse_labels is not None
                and hasattr(model, "coarse_head")
            ):
                y_coarse = torch.as_tensor(
                    coarse_labels[
                        start:start + batch_size
                    ],
                    device=device
                )

                coarse_logits = model.coarse_head(
                    head_features
                )

                coarse_pred = (
                    coarse_logits
                    .argmax(dim=1)
                )

                all_coarse_correct += (
                    coarse_pred == y_coarse
                ).sum().item()

                all_coarse_count += y_coarse.size(0)

        if all_probs:
            probs = torch.cat(
                all_probs,
                dim=0
            ).numpy()

            pred = probs.argmax(
                axis=1
            )

            mean_p_inlier = float(
                probs[:, 0].mean()
            )
            mean_p_outlier = float(
                probs[:, 1].mean()
            )
            mean_p_noise = float(
                probs[:, 2].mean()
            )

            frac_inlier = float(
                (pred == 0).mean()
            )
            frac_outlier = float(
                (pred == 1).mean()
            )
            frac_noise = float(
                (pred == 2).mean()
            )
        else:
            mean_p_inlier = np.nan
            mean_p_outlier = np.nan
            mean_p_noise = np.nan

            frac_inlier = np.nan
            frac_outlier = np.nan
            frac_noise = np.nan

        cosine_values = torch.cat(
            all_cosine,
            dim=0
        ).numpy()

        mean_cosine = float(
            cosine_values.mean()
        )

        fine_accuracy = (
            float(
                all_fine_correct
                / all_fine_count
            )
            if all_fine_count > 0
            else np.nan
        )

        coarse_accuracy = (
            float(
                all_coarse_correct
                / all_coarse_count
            )
            if all_coarse_count > 0
            else np.nan
        )

        alpha_bar = float(
            alpha_bars[step_idx]
        )

        signal_fraction = float(
            np.sqrt(alpha_bar)
        )

        noise_fraction = float(
            np.sqrt(1.0 - alpha_bar)
        )

        results["step"].append(step_idx)
        results["alpha_bar"].append(alpha_bar)
        results["signal_fraction"].append(
            signal_fraction
        )
        results["noise_fraction"].append(
            noise_fraction
        )

        results["mean_p_inlier"].append(
            mean_p_inlier
        )
        results["mean_p_outlier"].append(
            mean_p_outlier
        )
        results["mean_p_noise"].append(
            mean_p_noise
        )

        results["predicted_inlier_fraction"].append(
            frac_inlier
        )
        results["predicted_outlier_fraction"].append(
            frac_outlier
        )
        results["predicted_noise_fraction"].append(
            frac_noise
        )

        results["mean_clean_cosine_similarity"].append(
            mean_cosine
        )

        results["fine_accuracy"].append(
            fine_accuracy
        )
        results["coarse_accuracy"].append(
            coarse_accuracy
        )

    return results


# plottings

In [79]:

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


def _publication_style():
    plt.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.25,
    })


def save_publication_figure(fig, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


def save_manuscript_results(results, output_dir):
    """Save only paper-facing metrics. No trajectory data is stored here."""
    path = os.path.join(output_dir, "manuscript_results.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    return path


def manuscript_result_tables(results):
    """Return compact DataFrames for notebook display."""
    rep_order = [
        ("KNN", "Fine", "knn_fine"),
        ("Linear", "Fine", "linear_fine"),
        ("Angular", "Fine", "angular_fine"),
        ("KNN", "Coarse", "knn_coarse"),
        ("Linear", "Coarse", "linear_coarse"),
        ("Angular", "Coarse", "angular_coarse"),
    ]

    rep_rows = []
    for probe, level, key in rep_order:
        m = results["representation"][key]
        rep_rows.append({
            "Probe": probe,
            "Level": level,
            "Accuracy": m["accuracy"],
            "Macro F1": m["f1_macro"],
        })

    inlier_rows = []
    for level in ["fine", "coarse"]:
        if level in results["inlier_classification"]:
            m = results["inlier_classification"][level]
            inlier_rows.append({
                "Level": level.capitalize(),
                "Accuracy": m["accuracy"],
                "Macro F1": m["f1_macro"],
            })

    ood_rows = []
    for name in ["CIFAR10", "SVHN"]:
        d = results["ood"][name]
        g = d["generic_ood_score"]
        s = d["specific_outlier_probability_score"]
        t = results["three_class"][name]
        ood_rows.append({
            "Dataset": name,
            "Relation": d["relation"],
            "Seen": d["seen_during_probe_training"],
            "AUROC": g["AUROC"],
            "AUPR-OOD": g["AUPR_OOD"],
            "FPR@95": g["FPR_at_95_OOD_recall"],
            "P(outlier) AUROC": s["AUROC"],
            "3-class Accuracy": t["accuracy"],
            "3-class Macro F1": t["f1_macro"],
        })

    noise_rows = []
    for name in ["CIFAR10", "SVHN"]:
        m = results["noise"][name]
        noise_rows.append({
            "Dataset": name,
            "AUROC": m["AUROC"],
            "AUPR-OOD": m["AUPR_OOD"],
            "FPR@95": m["FPR_at_95_OOD_recall"],
        })

    return {
        "representation": pd.DataFrame(rep_rows),
        "inlier_classification": pd.DataFrame(inlier_rows),
        "ood": pd.DataFrame(ood_rows),
        "noise": pd.DataFrame(noise_rows),
    }


def display_manuscript_results(results):
    tables = manuscript_result_tables(results)
    for title, table in tables.items():
        print(f"\n{title.replace('_', ' ').title()}")
        try:
            from IPython.display import display
            display(table.round(4))
        except Exception:
            print(table.round(4).to_string(index=False))
    return tables


In [80]:

def plot_representation_results(results, output_dir):
    _publication_style()
    rep = results["representation"]

    probes = ["KNN", "Linear", "Angular"]
    fine = [
        rep["knn_fine"]["accuracy"],
        rep["linear_fine"]["accuracy"],
        rep["angular_fine"]["accuracy"],
    ]
    coarse = [
        rep["knn_coarse"]["accuracy"],
        rep["linear_coarse"]["accuracy"],
        rep["angular_coarse"]["accuracy"],
    ]

    x = np.arange(len(probes))
    width = 0.36
    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    ax.bar(x - width / 2, fine, width, label="Fine-grained")
    ax.bar(x + width / 2, coarse, width, label="Coarse-grained")

    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.set_xticks(x)
    ax.set_xticklabels(probes)
    ax.set_title("CIFAR-100 Frozen-Representation Classification")
    ax.grid(axis="y")
    ax.legend(frameon=False)
    fig.tight_layout()

    save_publication_figure(
        fig,
        os.path.join(output_dir, "representation_probe_comparison.png"),
    )


In [81]:

def plot_ood_results(results, output_dir):
    _publication_style()

    datasets = ["CIFAR10", "SVHN"]
    labels = ["CIFAR-10\nNear OOD", "SVHN\nFar OOD"]
    metrics = [
        ("AUROC", "AUROC"),
        ("AUPR_OOD", "AUPR-OOD"),
        ("FPR_at_95_OOD_recall", "FPR@95"),
    ]

    x = np.arange(len(datasets))
    width = 0.25
    fig, ax = plt.subplots(figsize=(9, 5.5))

    for i, (key, label) in enumerate(metrics):
        values = [
            results["ood"][dataset]["generic_ood_score"][key]
            for dataset in datasets
        ]
        ax.bar(x + (i - 1) * width, values, width, label=label)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.set_title("OOD Detection on Near and Far OOD")
    ax.grid(axis="y")
    ax.legend(frameon=False)
    fig.tight_layout()

    save_publication_figure(
        fig,
        os.path.join(output_dir, "near_far_ood_comparison.png"),
    )


In [82]:

def plot_three_class_confusion_matrices(results, output_dir):
    _publication_style()
    labels = ["Inlier", "OOD", "Noise"]
    names = [name for name in ["CIFAR10", "SVHN"] if name in results["three_class"]]
    if not names:
        return

    fig, axes = plt.subplots(
        1, len(names), figsize=(5.2 * len(names), 4.8), squeeze=False
    )
    axes = axes.ravel()

    for ax, name in zip(axes, names):
        cm = np.asarray(results["three_class"][name]["confusion_matrix"], dtype=float)
        row_sums = cm.sum(axis=1, keepdims=True)
        norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
        im = ax.imshow(norm)

        ax.set_xticks(np.arange(3))
        ax.set_yticks(np.arange(3))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(name)

        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center")

        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Row-normalized fraction")

    fig.suptitle("Three-Class Inlier / OOD / Noise Classification")
    fig.tight_layout()
    save_publication_figure(
        fig,
        os.path.join(output_dir, "three_class_confusion_matrices.png"),
    )


def plot_inlier_classification_results(results, output_dir):
    _publication_style()
    levels = [level for level in ["fine", "coarse"] if level in results["inlier_classification"]]
    if not levels:
        return

    x = np.arange(len(levels))
    width = 0.36
    accuracy = [results["inlier_classification"][l]["accuracy"] for l in levels]
    f1 = [results["inlier_classification"][l]["f1_macro"] for l in levels]

    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    ax.bar(x - width / 2, accuracy, width, label="Accuracy")
    ax.bar(x + width / 2, f1, width, label="Macro F1")
    ax.set_xticks(x)
    ax.set_xticklabels([l.capitalize() for l in levels])
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.set_title("CIFAR-100 Inlier Classification")
    ax.grid(axis="y")
    ax.legend(frameon=False)
    fig.tight_layout()

    save_publication_figure(
        fig,
        os.path.join(output_dir, "inlier_classification_results.png"),
    )


In [83]:

def save_noise_trajectory_results(trajectory_results, output_dir):
    """Save trajectory metrics outside manuscript_results.json."""
    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "noise_trajectory.json")
    csv_path = os.path.join(output_dir, "noise_trajectory.csv")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(trajectory_results, f, indent=2)

    pd.DataFrame(trajectory_results).to_csv(csv_path, index=False)
    return json_path, csv_path


def run_noise_trajectory_evaluation(cfg, exp):
    """
    Run the progressive clean -> Gaussian-noise trajectory independently
    from the manuscript result evaluation.
    """
    model = exp["model"]
    device = cfg["device"]
    batch_size = cfg.get("eval_batch_size", 256)

    print("\nPreparing 3-class probe for auxiliary noise trajectory...")

    _, in_train_head, _, _ = extract_eval_features(
        exp["in_train"], model, device,
        batch_size=batch_size, hierarchical=True,
    )
    _, out_train_head, _, _ = extract_eval_features(
        exp["out_train"], model, device,
        batch_size=batch_size, hierarchical=False,
    )
    _, noise_train_head, _, _ = extract_eval_features(
        exp["noise_train"], model, device,
        batch_size=batch_size, hierarchical=False,
    )

    probe = fit_three_class_probe(
        in_features=in_train_head,
        out_features=out_train_head,
        noise_features=noise_train_head,
        device=device,
        epochs=cfg.get("three_class_probe_epochs", 10),
        batch_size=batch_size,
        lr=cfg.get("three_class_probe_lr", 1e-3),
    )

    trajectory = evaluate_noise_trajectory(
        model=model,
        inlier_test_set=exp["in_test"],
        device=device,
        three_class_probe=probe,
        num_samples=cfg.get("noise_trajectory_samples", 2000),
        steps=cfg.get("noise_trajectory_steps", 10),
        batch_size=batch_size,
        final_alpha_bar=cfg.get("noise_trajectory_final_alpha_bar", 0.0),
        seed=cfg.get("noise_trajectory_seed", 42),
    )

    return trajectory


def plot_noise_trajectory(trajectory_results, output_dir):
    """Auxiliary figures. Not part of manuscript_results.json."""
    _publication_style()
    steps = np.asarray(trajectory_results["step"])
    x = np.asarray(trajectory_results.get("noise_fraction", steps), dtype=float)

    fig, ax = plt.subplots(figsize=(8, 5.2))
    ax.plot(x, trajectory_results["mean_p_inlier"], marker="o", linewidth=2, label="P(Inlier)")
    ax.plot(x, trajectory_results["mean_p_outlier"], marker="s", linewidth=2, label="P(Outlier)")
    ax.plot(x, trajectory_results["mean_p_noise"], marker="^", linewidth=2, label="P(Noise)")
    ax.set_xlabel("Injected noise fraction")
    ax.set_ylabel("Mean classifier probability")
    ax.set_ylim(0, 1)
    ax.set_xlim(0, 1)
    ax.set_title("Progressive Inlier / OOD / Noise Classification")
    ax.grid(axis="both")
    ax.legend(frameon=False)
    fig.tight_layout()
    save_publication_figure(
        fig, os.path.join(output_dir, "progressive_noise_rejection.png")
    )

    fig, ax = plt.subplots(figsize=(8, 5.2))
    ax.plot(
        x,
        trajectory_results["mean_clean_cosine_similarity"],
        marker="o", linewidth=2.2,
    )
    ax.set_xlabel("Injected noise fraction")
    ax.set_ylabel("Cosine similarity to clean embedding")
    ax.set_ylim(-1, 1)
    ax.set_xlim(0, 1)
    ax.set_title("Embedding Similarity Under Progressive Noise")
    ax.grid(axis="both")
    fig.tight_layout()
    save_publication_figure(
        fig, os.path.join(output_dir, "embedding_similarity_noise_trajectory.png")
    )

    fig, ax = plt.subplots(figsize=(8, 5.2))
    fine = np.asarray(trajectory_results["fine_accuracy"], dtype=float)
    coarse = np.asarray(trajectory_results["coarse_accuracy"], dtype=float)
    ax.plot(x, fine, marker="o", linewidth=2, label="Fine classification")
    ax.plot(x, coarse, marker="s", linewidth=2, label="Coarse classification")
    ax.set_xlabel("Injected noise fraction")
    ax.set_ylabel("Classification accuracy")
    ax.set_ylim(0, 1)
    ax.set_xlim(0, 1)
    ax.set_title("CIFAR-100 Classification Under Progressive Noise")
    ax.grid(axis="y")
    ax.legend(frameon=False)
    fig.tight_layout()
    save_publication_figure(
        fig, os.path.join(output_dir, "semantic_robustness_noise_trajectory.png")
    )


def _sample_indices(n, max_samples, rng):
    if n <= max_samples:
        return np.arange(n)
    return rng.choice(n, size=max_samples, replace=False)


def _fit_2d_embedding(features, method="tsne", random_state=42, perplexity=30):
    """PCA -> 2D or PCA(50) -> t-SNE for stable visualization."""
    features = np.asarray(features, dtype=np.float32)

    if features.shape[0] < 3:
        raise ValueError("Need at least 3 samples for embedding visualization.")

    if method.lower() == "pca":
        return PCA(n_components=2, random_state=random_state).fit_transform(features)

    n_pca = min(50, features.shape[1], features.shape[0] - 1)
    reduced = PCA(n_components=n_pca, random_state=random_state).fit_transform(features)
    perplexity = min(perplexity, max(5, (features.shape[0] - 1) // 3))

    return TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=1000,
        random_state=random_state,
    ).fit_transform(reduced)


def _scatter_labeled(ax, xy, labels, title, colorbar_label=None, cmap="turbo"):
    labels = np.asarray(labels)
    sc = ax.scatter(
        xy[:, 0], xy[:, 1],
        c=labels,
        s=10,
        alpha=0.65,
        linewidths=0,
        cmap=cmap,
    )
    ax.set_title(title)
    ax.set_xlabel("Embedding dimension 1")
    ax.set_ylabel("Embedding dimension 2")
    if colorbar_label is not None:
        cbar = ax.figure.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(colorbar_label)
    ax.grid(alpha=0.15)


def _scatter_cdsh(ax, xy, labels):
    labels = np.asarray(labels)
    class_names = ["Inlier", "OOD", "Noise"]
    markers = ["o", "s", "^"]
    for class_id, (name, marker) in enumerate(zip(class_names, markers)):
        mask = labels == class_id
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            s=12,
            alpha=0.65,
            marker=marker,
            label=name,
            linewidths=0,
        )
    ax.set_title("C_DSH: Inlier / OOD / Noise")
    ax.set_xlabel("Embedding dimension 1")
    ax.set_ylabel("Embedding dimension 2")
    ax.grid(alpha=0.15)
    ax.legend(frameon=False)


def collect_embedding_visualization_data(cfg, exp, max_inlier=3000, max_per_cdsh_class=1000, seed=42):
    """
    Collect frozen UHCL embeddings for publication visualization.

    The primary embedding is model output `z` (not the classifier-head feature).
    """
    model = exp["model"]
    device = cfg["device"]
    batch_size = cfg.get("eval_batch_size", 256)
    rng = np.random.default_rng(seed)

    test_transform = get_transforms("cifar100", "test")
    cifar10_test = datasets.CIFAR10(
        "./data", train=False, transform=test_transform, download=True
    )
    svhn_test = datasets.SVHN(
        "./data", split="test", transform=test_transform, download=True
    )

    inlier_z, _, fine_labels, coarse_labels = extract_eval_features(
        exp["in_test"], model, device, batch_size=batch_size, hierarchical=True
    )
    cifar10_z, _, _, _ = extract_eval_features(
        cifar10_test, model, device, batch_size=batch_size, hierarchical=False
    )
    svhn_z, _, _, _ = extract_eval_features(
        svhn_test, model, device, batch_size=batch_size, hierarchical=False
    )
    noise_z, _, _, _ = extract_eval_features(
        exp["noise_test"], model, device, batch_size=batch_size, hierarchical=False
    )

    idx_inlier = _sample_indices(len(inlier_z), max_inlier, rng)
    fine_sample = fine_labels[idx_inlier]
    coarse_sample = coarse_labels[idx_inlier]
    inlier_features = inlier_z[idx_inlier]

    class_arrays = [inlier_z, np.concatenate([cifar10_z, svhn_z], axis=0), noise_z]
    sampled = []
    labels = []
    for class_id, arr in enumerate(class_arrays):
        idx = _sample_indices(len(arr), max_per_cdsh_class, rng)
        sampled.append(arr[idx])
        labels.append(np.full(len(idx), class_id, dtype=np.int64))

    cdsh_features = np.concatenate(sampled, axis=0)
    cdsh_labels = np.concatenate(labels, axis=0)

    return {
        "inlier_features": inlier_features,
        "fine_labels": fine_sample,
        "coarse_labels": coarse_sample,
        "cdsh_features": cdsh_features,
        "cdsh_labels": cdsh_labels,
    }


def plot_embedding_spaces(
    cfg,
    exp,
    output_dir,
    method="tsne",
    max_inlier=3000,
    max_per_cdsh_class=1000,
    seed=42,
):
    """
    Publication-quality embedding visualizations:

      1. CIFAR-100 inliers colored by fine class.
      2. The SAME CIFAR-100 projection colored by coarse class.
      3. Full C_DSH colored by Inlier / OOD / Noise.

    OOD combines CIFAR-10 and SVHN, matching the 3-class C_DSH definition.
    """
    data = collect_embedding_visualization_data(
        cfg,
        exp,
        max_inlier=max_inlier,
        max_per_cdsh_class=max_per_cdsh_class,
        seed=seed,
    )

    emb_dir = os.path.join(output_dir, "embeddings")
    os.makedirs(emb_dir, exist_ok=True)

    # One projection for both fine and coarse labels.
    inlier_xy = _fit_2d_embedding(
        data["inlier_features"], method=method, random_state=seed
    )

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    _scatter_labeled(
        ax,
        inlier_xy,
        data["fine_labels"],
        "CIFAR-100 Inlier Embedding — Fine Classes",
        colorbar_label="Fine-class ID",
    )
    fig.tight_layout()
    save_publication_figure(
        fig,
        os.path.join(emb_dir, "embedding_inliers_fine.png"),
    )

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    _scatter_labeled(
        ax,
        inlier_xy,
        data["coarse_labels"],
        "CIFAR-100 Inlier Embedding — Coarse Classes",
        colorbar_label="Coarse-class ID",
        cmap="tab20",
    )
    fig.tight_layout()
    save_publication_figure(
        fig,
        os.path.join(emb_dir, "embedding_inliers_coarse.png"),
    )

    cdsh_xy = _fit_2d_embedding(
        data["cdsh_features"], method=method, random_state=seed
    )
    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    _scatter_cdsh(ax, cdsh_xy, data["cdsh_labels"])
    fig.tight_layout()
    save_publication_figure(
        fig,
        os.path.join(emb_dir, "embedding_cdsh_inlier_ood_noise.png"),
    )

    # Save coordinates separately so plots can be re-rendered without feature extraction.
    np.savez_compressed(
        os.path.join(emb_dir, "embedding_visualization_coordinates.npz"),
        inlier_xy=inlier_xy,
        fine_labels=data["fine_labels"],
        coarse_labels=data["coarse_labels"],
        cdsh_xy=cdsh_xy,
        cdsh_labels=data["cdsh_labels"],
    )

    return data


#  RUN

In [84]:
# if __name__ == "__main__":

#     print("Starting UHCL updated runner")

#     (
#         model,
#         in_train,
#         in_test,
#         out_train,
#         out_test,
#         noise_train,
#         noise_test,
#     ) = train(config)

    

In [85]:

print("Starting UHCL manuscript evaluation")

checkpoint_path = config["resume_ckpt_path"]
exp = load_checkpoint_for_evaluation(config, checkpoint_path)
results = evaluate_manuscript(config, exp)

output_dir = os.path.join(
    config.get("results_dir", "./results"),
    config["experiment_name"],
)
os.makedirs(output_dir, exist_ok=True)

results_path = save_manuscript_results(results, output_dir)
print("\nEvaluation complete.")
print("Saved manuscript results:", results_path)

# Compact notebook display instead of printing a deeply nested dict.
tables = display_manuscript_results(results)


In [86]:

required_result_keys = {
    "representation",
    "inlier_classification",
    "ood",
    "three_class",
    "noise",
}

missing = required_result_keys - set(results.keys())
if missing:
    raise RuntimeError(
        "Evaluation results are incomplete. Missing keys: "
        + ", ".join(sorted(missing))
    )

plot_representation_results(results, output_dir)
plot_inlier_classification_results(results, output_dir)
plot_ood_results(results, output_dir)
plot_three_class_confusion_matrices(results, output_dir)

print("\nManuscript figures written to:")
print(output_dir)


In [87]:

# ============================================================
# OPTIONAL AUXILIARY EVALUATIONS
# ============================================================
# These do NOT modify `results` and are saved separately.
# Set the switches to True when you want to run them.

RUN_NOISE_TRAJECTORY = True
RUN_EMBEDDING_PLOTS = True


In [88]:

if RUN_NOISE_TRAJECTORY:
    auxiliary_dir = os.path.join(output_dir, "auxiliary")
    trajectory_results = run_noise_trajectory_evaluation(config, exp)
    json_path, csv_path = save_noise_trajectory_results(
        trajectory_results,
        auxiliary_dir,
    )
    plot_noise_trajectory(
        trajectory_results,
        auxiliary_dir,
    )
    print("\nNoise trajectory saved to:")
    print(json_path)
    print(csv_path)


In [89]:

if RUN_EMBEDDING_PLOTS:
    plot_embedding_spaces(
        config,
        exp,
        output_dir,
        method="tsne",
        max_inlier=3000,
        max_per_cdsh_class=1000,
        seed=config.get("noise_trajectory_seed", 42),
    )
    print("\nEmbedding plots saved to:")
    print(os.path.join(output_dir, "embeddings"))
